# Ranking and top-N per group

Practice notebook for the datafromzero SQL lesson **Ranking and top-N per group**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Orders per customer and cafe: look at Harbor's top rows

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
)
SELECT customer_id, orders
FROM per_customer
WHERE store_id = 3
ORDER BY orders DESC, customer_id
LIMIT 8
""").show()

### Step 2. Three ranking functions on the same ties (predict RANK for the last three rows first)

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
)
SELECT customer_id, orders,
       ROW_NUMBER() OVER (ORDER BY orders DESC, customer_id) AS row_num,
       RANK()       OVER (ORDER BY orders DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY orders DESC) AS dense_rnk
FROM per_customer
WHERE store_id = 3
ORDER BY orders DESC, customer_id
LIMIT 8
""").show()

### Step 3. The top 3 customers of each cafe, exactly three per cafe

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
),
ranked AS (
  SELECT store_id, customer_id, orders,
         ROW_NUMBER() OVER (PARTITION BY store_id
                            ORDER BY orders DESC, customer_id) AS row_num
  FROM per_customer
)
SELECT store_id, customer_id, orders, row_num
FROM ranked
WHERE row_num <= 3
ORDER BY store_id, row_num
""").show()

### Step 4. Why the filter needs a second stage (this cell gives an error on purpose)

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
)
SELECT store_id, customer_id, orders,
       ROW_NUMBER() OVER (PARTITION BY store_id
                          ORDER BY orders DESC, customer_id) AS row_num
FROM per_customer
WHERE row_num <= 3
""").show()

### Step 5. The same filter with RANK: ties stay

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
),
ranked AS (
  SELECT store_id, customer_id, orders,
         RANK() OVER (PARTITION BY store_id ORDER BY orders DESC) AS rnk
  FROM per_customer
)
SELECT store_id, customer_id, orders, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY store_id, rnk, customer_id
""").show()

### Step 6. Verify: one row per group

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
),
ranked AS (
  SELECT store_id, customer_id, orders,
         ROW_NUMBER() OVER (PARTITION BY store_id
                            ORDER BY orders DESC, customer_id) AS row_num
  FROM per_customer
),
top3 AS (
  SELECT store_id, customer_id, orders, row_num
  FROM ranked
  WHERE row_num <= 3
)
SELECT store_id, COUNT(*) AS rows_kept
FROM top3
GROUP BY store_id
ORDER BY store_id
""").show()

### Step 7. The DuckDB shortcut: QUALIFY

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
)
SELECT store_id, customer_id, orders,
       ROW_NUMBER() OVER (PARTITION BY store_id
                          ORDER BY orders DESC, customer_id) AS row_num
FROM per_customer
QUALIFY row_num <= 3
ORDER BY store_id, row_num
""").show()

## Exercise 1. The three best-selling products

The `per_product` stage and a query that lists it are given. Rank the products by units sold and keep the top 3.
Products appear as ids; the lesson says where the names come from.

In [ ]:
con.sql("""
WITH per_product AS (
  SELECT product_id, SUM(quantity) AS units
  FROM order_items
  GROUP BY product_id
)
SELECT product_id, units
FROM per_product
ORDER BY units DESC
""").show()

## Exercise 2. The most frequent customer for each payment method

Before you run anything, write one sentence in the first cell: how many rows do you expect, and what do you do with a
tie for first place? Then write the query. Keep every customer who ties for first.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

The owner has exactly three gift cards for each cafe. A colleague runs this query for the winners. It runs, but
something is wrong with the number of rows per cafe. Find it and write the fix.

In [ ]:
con.sql("""
WITH per_customer AS (
  SELECT store_id, customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY store_id, customer_id
),
ranked AS (
  SELECT store_id, customer_id, orders,
         RANK() OVER (PARTITION BY store_id ORDER BY orders DESC) AS rnk
  FROM per_customer
)
SELECT store_id, customer_id, orders
FROM ranked
WHERE rnk <= 3
ORDER BY store_id, orders DESC, customer_id
""").show()

In [ ]:
# Your corrected query: